# 05 — Pembandingan Vs30 dengan model mHVSR

**Alur:** `HVSR teramati → prediksi Vs30 mHVSR; Vs30 profil final → perbandingan akhir`.

**Input:** kurva HVSR dari notebook 01, status Vs30 dari notebook 04, dan path bobot model pembanding pada sel di bawah.

**Proses:** ubah kurva HVSR menjadi fitur sesuai kontrak model mHVSR, hitung prediksi Vs30, lalu bandingkan dengan Vs30 dari profil lapisan bila hasil tersebut sudah final. Prediksi pembanding tidak mengubah inversi atau profil Vs.

**Output:** prediksi, fitur, dan metadata di `outputs/<site_id>/05/`. Tabel selisih serta grafik perbandingan hanya dibuat bila Vs30 profil telah dinyatakan final; status QC HVSR tetap ditampilkan.


## Input pengguna — lokasi model pembanding

Biarkan path bawaan bila model lama masih ada di folder `models/`. `SITE_ID` harus sama dengan yang diisi pada notebook 01–04. Berkas MiniSEED tidak dibaca ulang oleh notebook ini.

In [ ]:
SITE_ID = "solo_pilot"
MODEL_RELATIVE_PATH = "models/log_ANN_model.keras"

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mhvsr_vs30.mam.legacy_mhvsr import predict_single_input_mhvsr, resample_hvsr_features

ROOT = Path.cwd()
SITE = SITE_ID
SITE_INPUTS = json.loads((ROOT / 'outputs' / SITE / '01' / 'site_inputs.json').read_text(encoding='utf-8'))
assert SITE_INPUTS['site_id'] == SITE
P01 = ROOT / 'outputs' / SITE / '01'
P04 = ROOT / 'outputs' / SITE / '04'
OUT = ROOT / 'outputs' / SITE / '05'
OUT.mkdir(parents=True, exist_ok=True)
REFINEMENT_STATUS = P04 / 'refinement_status.json'
FINAL_VS30 = P04 / 'vs30_results.csv'
HVSR_INPUT = P01 / 'hvsr_observed.csv'
MODEL_INPUT = ROOT / MODEL_RELATIVE_PATH
PREDICTION_INPUT = OUT / 'prediction_vs30_mhvsr.csv'
refinement_check=json.loads(REFINEMENT_STATUS.read_text(encoding='utf-8'))
if refinement_check.get('run_state')!='complete':
    raise ValueError('Notebook 04 must complete before comparison')
if hashlib.sha256((ROOT/'outputs'/SITE/'03'/'inversion_status.json').read_bytes()).hexdigest()!=refinement_check.get('inversion_status_sha256'):
    raise ValueError('Inversion changed: rerun notebook 04 before comparison')
if hashlib.sha256(HVSR_INPUT.read_bytes()).hexdigest()!=refinement_check.get('hvsr_source_sha256'):
    raise ValueError('HVSR changed: rerun notebook 04 before comparison')
inversion_check=json.loads((ROOT/'outputs'/SITE/'03'/'inversion_status.json').read_text(encoding='utf-8'))
for artifact,expected in ((ROOT/inversion_check['source_csv'],inversion_check['source_sha256']),
    (ROOT/'outputs'/SITE/'02'/'dispersion_qc.json',inversion_check['upstream_qc_sha256'])):
    if hashlib.sha256(artifact.read_bytes()).hexdigest()!=expected:
        raise ValueError('Dispersion changed: rerun notebooks 03-04 before comparison')
for name in ('vs30_comparison.csv','vs30_comparison.png'):
    old=OUT/name
    if old.exists():
        archive=OUT/'superseded'/datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')
        archive.mkdir(parents=True,exist_ok=True)
        old.replace(archive/name)


## 1. Jalankan model mHVSR lama sebagai pembanding

Praproses mengikuti `high_dim_models.ipynb`: interpolasi linier 35 titik logaritmik dan logaritma natural amplitudo. Model tersimpan dihitung pada mode inferensi; bobot dan masukan dicatat checksum-nya.


In [ ]:
hvsr = pd.read_csv(HVSR_INPUT)
frequency, log_amplitude = resample_hvsr_features(
    hvsr.frequency_hz.to_numpy(), hvsr.hvsr_mean.to_numpy()
)
predicted_vs30 = predict_single_input_mhvsr(MODEL_INPUT, log_amplitude)
refinement = json.loads(REFINEMENT_STATUS.read_text(encoding='utf-8'))
prediction_qc = (
    'provisional_hvsr_clarity_failed'
    if 'sesame_clarity_failed' in refinement['hvsr_qc_flags']
    else 'hvsr_clarity_passed'
)
pd.DataFrame({
    'frequency_hz': frequency,
    'hvsr_amplitude': np.exp(log_amplitude),
    'log_hvsr_amplitude': log_amplitude,
}).to_csv(OUT / 'legacy_mhvsr_features.csv', index=False)
pd.DataFrame([{'site_id': SITE,
    'model_name': 'published_single_input_mhvsr_ann',
    'predicted_vs30_m_s': predicted_vs30,
    'provenance': f'high_dim_models.ipynb + {MODEL_RELATIVE_PATH}',
    'prediction_qc': prediction_qc,
}]).to_csv(PREDICTION_INPUT, index=False)
prediction_metadata = {
    'site_id': SITE, 'created_at_utc': datetime.now(UTC).isoformat(),
    'model_path': str(MODEL_INPUT.relative_to(ROOT)),
    'model_sha256': hashlib.sha256(MODEL_INPUT.read_bytes()).hexdigest(),
    'hvsr_path': str(HVSR_INPUT.relative_to(ROOT)),
    'hvsr_sha256': hashlib.sha256(HVSR_INPUT.read_bytes()).hexdigest(),
    'feature_count': len(frequency), 'feature_band_hz': [0.3, 50.0],
    'feature_transform': 'linear amplitude interpolation then natural log, float32',
    'output_transform': 'exp of predicted log Vs30',
    'site_elevation_m_context_only': SITE_INPUTS['station_elevation_m'][SITE_INPUTS['hvsr_station']],
    'prediction_qc': prediction_qc,
    'role': 'final_comparator_only',
}
(OUT / 'prediction_metadata.json').write_text(
    json.dumps(prediction_metadata, indent=2), encoding='utf-8'
)
print(f'mHVSR comparator: {predicted_vs30:.2f} m/s ({prediction_qc})')

## 2. Bandingkan hanya jika Vs30 hasil inversi sudah final

Nilai prediksi harus positif dan berasal dari site yang sama. Selisih `Vs30_final − Vs30_prediksi` dinyatakan dalam m/s dan persen terhadap Vs30 final. Jika Vs30 final belum tersedia, notebook hanya menyimpan status tanpa tabel perbandingan.


In [ ]:
refinement=json.loads(REFINEMENT_STATUS.read_text(encoding='utf-8'))
prediction=pd.read_csv(PREDICTION_INPUT)
blockers=[]
if not refinement['vs30_reported'] or not FINAL_VS30.is_file():
    blockers.append('vs30_final_not_available_from_notebook_04')
if prediction.empty:
    blockers.append('prediction_vs30_mhvsr_empty')
else:
    required={'site_id','model_name','predicted_vs30_m_s','provenance'}
    if not required.issubset(prediction.columns):
        blockers.append('prediction_input_columns_invalid')
    elif (prediction.predicted_vs30_m_s.isna().any() or
          (prediction.predicted_vs30_m_s<=0).any() or
          prediction.model_name.isna().any() or prediction.provenance.isna().any() or
          not prediction.site_id.eq(SITE).all()):
        blockers.append('prediction_input_values_invalid')

comparison=None
if not blockers:
    if hashlib.sha256(FINAL_VS30.read_bytes()).hexdigest()!=refinement.get('vs30_results_sha256'):
        raise ValueError('Final Vs30 changed: rerun notebook 04 before comparison')
    final=pd.read_csv(FINAL_VS30)
    assert len(final)==1 and final.site_id.iloc[0]==SITE
    final_value=float(final.vs30_m_s.iloc[0])
    assert np.isfinite(final_value) and final_value>0
    comparison=prediction.copy()
    comparison['final_vs30_m_s']=final_value
    comparison['difference_final_minus_prediction_m_s']=final_value-comparison.predicted_vs30_m_s
    comparison['difference_percent_of_final']=100*(final_value-comparison.predicted_vs30_m_s)/final_value
    comparison.to_csv(OUT/'vs30_comparison.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,4))
    ax.axhline(final_value,color='black',label='Vs30 final')
    ax.scatter(comparison.model_name,comparison.predicted_vs30_m_s,label='Legacy prediction')
    ax.set(ylabel='Vs30 (m/s)',title=f'Final Vs30 vs prediction — {SITE}')
    ax.grid(alpha=0.2); ax.legend(); fig.tight_layout()
    fig.savefig(OUT/'vs30_comparison.png',dpi=160); plt.show()

status={'site_id':SITE,'created_at_utc':datetime.now(UTC).isoformat(),
    'comparison_complete':comparison is not None,'blocking_reasons':blockers,
    'prediction_role':'final_comparator_only',
    'prediction_changes_final_vs':False,
    'refinement_status_sha256':hashlib.sha256(REFINEMENT_STATUS.read_bytes()).hexdigest(),
    'prediction_input_sha256':hashlib.sha256(PREDICTION_INPUT.read_bytes()).hexdigest(),
    'prediction_qc':prediction_qc}
(OUT/'comparison_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('Comparison complete:',status['comparison_complete'],'| blockers:',blockers)
